# Train CoDy-JEPA on Health&Gait

This notebook is an interactive front end to the maintained training implementation under `src/cody_jepa`. It does not copy or redefine the model, masking, loss, optimization, evaluation, or checkpoint functions. Importing those functions keeps the command-line and notebook paths on the same implementation.

## Execution contract

A full run requires the private Health&Gait release, its subject-disjoint manifest, and the CUDA device required by the checked-in baseline configuration. The training cell writes `latest.pt`, `best_loss.pt`, and, when the representation-health criteria are met, `best_healthy.pt`.

For reproducible execution, start Jupyter through the repository environment, restart the kernel, and run all cells in order. Do not inspect a batch from the shuffled training loader before the training cell because doing so advances its random generator.

In [ ]:
from pathlib import Path
import inspect
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

from cody_jepa.cli.train import _read_config
from cody_jepa.data import (
    HealthGaitLoaderConfig,
    build_healthgait_datasets_from_config,
    build_healthgait_loaders_from_config,
)
from cody_jepa.training import load_checkpoint, train_jepa, validate_training_config


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/cody_jepa").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the cody-jepa repository")


REPO_ROOT = find_repo_root()
training_source = Path(inspect.getsourcefile(train_jepa)).resolve()
expected_source = (REPO_ROOT / "src/cody_jepa/training/engine.py").resolve()
if training_source != expected_source:
    raise RuntimeError(
        f"Imported train_jepa from {training_source}, expected repository source {expected_source}"
    )

print(f"Repository: {REPO_ROOT}")
print(f"Training implementation: {training_source}")
print(f"PyTorch: {torch.__version__}")

## Run parameters

The loader values below mirror the defaults in `cody_jepa.cli.train`. Change an override only when intentionally defining a different run. Use a new output directory to avoid replacing an existing checkpoint set.

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs/train/healthgait_baseline.json"
MANIFEST_PATH = (
    REPO_ROOT
    / "data/healthgait/manifests/silhouette_subject_split_seed0.csv"
)
OUTPUT_DIR = REPO_ROOT / "outputs/notebook-training-baseline"

RESUME_PATH = None
DEVICE_OVERRIDE = None
BATCH_SIZE_OVERRIDE = None

NUM_WORKERS = 4
EVAL_WINDOWS = 3
IMAGE_VERIFY_MODE = "sample"
DROP_LAST_TRAIN = True

for required_path in (CONFIG_PATH, MANIFEST_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)
if RESUME_PATH is not None and not Path(RESUME_PATH).is_file():
    raise FileNotFoundError(RESUME_PATH)

print(f"Configuration: {CONFIG_PATH.relative_to(REPO_ROOT)}")
print(f"Manifest: {MANIFEST_PATH.relative_to(REPO_ROOT)}")
print(f"Output directory: {OUTPUT_DIR.relative_to(REPO_ROOT)}")

## Resolve the training configuration and mask preset

The CLI resolver validates the selected mask preset and constructs the same immutable mask-group objects passed by the command-line entry point.

In [ ]:
config, mask_groups = _read_config(CONFIG_PATH)
if DEVICE_OVERRIDE is not None:
    config["required_device"] = DEVICE_OVERRIDE
if BATCH_SIZE_OVERRIDE is not None:
    config["batch_size"] = BATCH_SIZE_OVERRIDE

mask_table = pd.DataFrame(
    [
        {
            "label": group.label,
            "num_blocks": group.num_blocks,
            "spatial_scale": group.spatial_scale,
            "aspect_ratio": group.aspect_ratio,
        }
        for group in mask_groups
    ]
)
print(json.dumps(config, indent=2))
display(mask_table)

## Build the canonical datasets and loaders

This cell reproduces the CLI's loader construction exactly. It validates the manifest and image boundary, creates random-window training data, creates deterministic validation windows, seeds the shuffled training loader, and drops the final incomplete training batch.

In [ ]:
loader_config = HealthGaitLoaderConfig(
    manifest_csv=MANIFEST_PATH,
    repo_root=REPO_ROOT,
    split="train",
    clip_length=int(config["num_frames"]),
    image_size=(int(config["img_size"]), int(config["img_size"])),
    channels=int(config["in_channels"]),
    seed=int(config.get("seed", 0)),
    batch_size=int(config["batch_size"]),
    num_workers=NUM_WORKERS,
    pin_memory=(
        config.get("required_device") == "cuda" or torch.cuda.is_available()
    ),
    prefetch_factor=1,
    train_crop_scale=(0.90, 1.0),
    train_horizontal_flip_prob=float(config["train_horizontal_flip_prob"]),
    strict_frame_sequence=True,
    image_verify_mode=IMAGE_VERIFY_MODE,
    allowed_data_root=REPO_ROOT / "data/healthgait",
    eval_windows=EVAL_WINDOWS,
    drop_last_train=DROP_LAST_TRAIN,
)
datasets = build_healthgait_datasets_from_config(loader_config)
train_loader, val_loader = build_healthgait_loaders_from_config(
    loader_config, datasets=datasets
)

dataset_summary = pd.DataFrame(
    [
        {
            "split": dataset.split,
            "examples": len(dataset),
            "batches": len(train_loader if dataset.split == "train" else val_loader),
        }
        for dataset in datasets
    ]
)
display(dataset_summary)

## Freeze the resume contract

Checkpoints store this path-independent description of the loaders and datasets. Resume is rejected if the scientific data or loader contract differs, while machine-specific absolute paths are intentionally excluded.

In [ ]:
data_contract = {
    "loader_config": {
        key: value
        for key, value in loader_config.as_dict().items()
        if key not in {"manifest_csv", "repo_root", "allowed_data_root"}
    },
    "train_dataset": datasets[0].description(),
    "val_dataset": datasets[1].description(),
}

updates_per_epoch = validate_training_config(config, train_loader)
print(json.dumps(data_contract, indent=2))
print(f"Updates per epoch: {updates_per_epoch}")
print(f"Configured optimizer steps: {config['steps']}")

## Train or resume

The following call is the same canonical call made by `cody_jepa.cli.train`. Model construction, random seeding, masking, mixed precision, loss calculation, gradient accumulation, optimizer steps, EMA updates, validation, health checks, and atomic checkpoint writes all remain inside the package implementation.

In [ ]:
resume_state = (
    load_checkpoint(Path(RESUME_PATH).expanduser().resolve())
    if RESUME_PATH is not None
    else None
)
result = train_jepa(
    config,
    train_loader,
    val_loader,
    data_contract,
    checkpoint_dir=OUTPUT_DIR,
    resume_state=resume_state,
    device=config.get("required_device", "auto"),
    mask_groups=mask_groups,
)

## Inspect the result and checkpoints

Presentation starts only after training has completed, so it cannot advance the training loader or alter the saved model state.

In [ ]:
summary = {
    "output_dir": str(OUTPUT_DIR),
    "global_step": result["global_step"],
    "completed_epochs": result["completed_epochs"],
    "best_epoch": result["best_epoch"],
    "best_healthy_epoch": result["best_healthy_epoch"],
    "termination_reason": result["termination_reason"],
    "elapsed_seconds": result["elapsed_seconds"],
    "examples_per_second": result["examples_per_second"],
}
display(pd.Series(summary, name="value").to_frame())

checkpoint_table = pd.DataFrame(
    [
        {"checkpoint": name, "exists": (OUTPUT_DIR / name).is_file()}
        for name in ("latest.pt", "best_loss.pt", "best_healthy.pt")
    ]
)
display(checkpoint_table)

In [ ]:
history = pd.json_normalize(result["history"], sep=".")
if history.empty:
    raise RuntimeError("Training returned an empty history")
display(history.tail())

plot_columns = ["train_loss"]
if "val.loss" in history.columns:
    plot_columns.append("val.loss")
axis = history.plot(
    x="step",
    y=plot_columns,
    marker="o",
    figsize=(10, 5),
    title="JEPA training history",
)
axis.set_ylabel("Loss")
axis.grid(alpha=0.3)
plt.show()

## Reproducibility boundary

This notebook and the CLI execute the same Python function objects with the same resolved inputs. Wall-clock time, throughput, and peak-memory measurements will differ. Bitwise equality is not promised across different GPU models, CUDA builds, PyTorch versions, or kernels because the training engine seeds its random generators but does not enable PyTorch's strict deterministic-algorithm mode. Exact checkpoint resume is protected by the saved model, optimizer, scaler, loader, mask, Torch, and CUDA random states plus the validated configuration and data contract.